# Install and load necesary packages

In [2]:
# Please don't change this cell

import pandas as pd
import numpy as np  

import warnings
warnings.filterwarnings("ignore")

## Load the dataset using pandas

In [5]:
# Please don't change this cell
df = pd.read_csv('ml-100k/u.data', names=['user_id', 'item_id', 'rating', 'timestamp'], sep='\t')

df.head()

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


# Split dataset


## Randomly select one rating from each user as test set

In [6]:
# please do not change this cell

from sklearn.model_selection import train_test_split

n_users = df.user_id.unique().shape[0]
n_items = df.item_id.unique().shape[0]
print(str(n_users) + ' users')
print(str(n_items) + ' items')

train_df, test_df = train_test_split(df, test_size=0.2, random_state = 10)
train_df, test_df

# Training Dataset
train_ds = np.zeros((n_users, n_items))
item_popularity = np.zeros(n_items)
for row in train_df.itertuples():
    train_ds[row[1]-1, row[2]-1] = row[3]
    item_popularity[row[2]-1] =  item_popularity[row[2]-1] + 1
#train_ds = pd.DataFrame(train_ds)

# Testing Dataset
testsize = 0
test_ds = np.zeros((n_users, n_items))
for row in test_df.itertuples():
    if item_popularity[row[2]-1] > 30:
        test_ds[row[1]-1, row[2]-1] = row[3]
        testsize = testsize + 1
#test_ds = pd.DataFrame(test_ds)

print("Construct the rating matrix based on train_df:")
print(train_ds)

print("Construct the rating matrix based on test_df:")
print(test_ds)

print("Testsize = " + str(testsize))

943 users
1682 items
Construct the rating matrix based on train_df:
[[0. 3. 4. ... 0. 0. 0.]
 [4. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [5. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 5. 0. ... 0. 0. 0.]]
Construct the rating matrix based on test_df:
[[5. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
Testsize = 17678


# MAE and RMSE Utils

In [7]:
# Please don't change this cell
# you can use this devaluate Utils here, and you can also implement your own MAE and RMSE calculation. 

EPSILON = 1e-9

def evaluate(test_ds, predicted_ds):
    '''
    Function for evaluating on MAE and RMSE
    '''
    # MAE
    mask_test_ds = test_ds > 0
    MAE = np.sum(np.abs(test_ds[mask_test_ds] - predicted_ds[mask_test_ds])) / np.sum(mask_test_ds.astype(np.float32))

    # RMSE
    RMSE = np.sqrt(np.sum(np.square(test_ds[mask_test_ds] - predicted_ds[mask_test_ds])) / np.sum(mask_test_ds.astype(np.float32)))

    return MAE, RMSE

# Your Solution for Method 1

In [6]:
# Write your code here for Method 1
# You are required to implement the required solution 1 here. 
# Then, evaluate your implementation by predicting the ratings in the test set (test_ds).
# Finally, save the corresponding MAE and RMSE of your implementation 
# into the following defined corresponding variable. 

MAE_solution1 = 0 # 0 is an intial value, you need to update this with the actual perofrmance of your implementation.
RMSE_solution1 = 0 # 0 is an intial value, you need to update this with the actual perofrmance of your implementation.
# Method 1: Predict using the average rating of each user

# Define the evaluation function in case it hasn't been run yet
def evaluate(test_ds, predicted_ds):
    '''
    Function for evaluating MAE and RMSE between the predicted ratings and the test ratings.
    '''
    mask_test_ds = test_ds > 0
    MAE = np.sum(np.abs(test_ds[mask_test_ds] - predicted_ds[mask_test_ds])) / np.sum(mask_test_ds.astype(np.float32))
    RMSE = np.sqrt(np.sum(np.square(test_ds[mask_test_ds] - predicted_ds[mask_test_ds])) / np.sum(mask_test_ds.astype(np.float32)))
    return MAE, RMSE

# Step 1: Initialize the predicted rating matrix with zeros
predicted_ds = np.zeros((n_users, n_items))

# Step 2: Compute average rating for each user from the training dataset
user_avg_ratings = np.zeros(n_users)

for user_id in range(n_users):
    rated_items = train_ds[user_id, :] > 0
    if np.sum(rated_items) > 0:
        user_avg_ratings[user_id] = np.mean(train_ds[user_id, rated_items])
    else:
        user_avg_ratings[user_id] = 0  # If no ratings, set average to 0

# Step 3: Use user averages to fill in the predicted ratings for test entries
for user_id in range(n_users):
    for item_id in range(n_items):
        if test_ds[user_id, item_id] > 0:
            predicted_ds[user_id, item_id] = user_avg_ratings[user_id]

# Step 4: Evaluate prediction accuracy
MAE_solution1, RMSE_solution1 = evaluate(test_ds, predicted_ds)

# Print results (optional)
print("MAE (Method 1 - User Average):", MAE_solution1)
print("RMSE (Method 1 - User Average):", RMSE_solution1)



MAE (Method 1 - User Average): 0.8258905090105331
RMSE (Method 1 - User Average): 1.031143070595138


In [7]:
# Please don't change this cell

print("===================== The MAE and RMSE of Your Implementation =====================")
print("MAE: {}, RMSE: {}" .format(MAE_solution1, RMSE_solution1))

===================== The MAE and RMSE of Your Implementation =====================
MAE: 0.8258905090105331, RMSE: 1.031143070595138


# Your Solution for Method 2

In [8]:
# Write your code here for Method 2
# You are required to implement the required solution 1 here. 
# Then, evaluate your implementation by predicting the ratings in the test set (test_ds).
# Finally, save the corresponding MAE and RMSE of your implementation 
# into the following defined corresponding variable. 

MAE_solution2 = 0 # 0 is an intial value, you need to update this with the actual perofrmance of your implementation.
RMSE_solution2 = 0 # 0 is an intial value, you need to update this with the actual perofrmance of your implementation.
# Method 2: Predict using the average rating of each item

# Define the evaluation function in case it hasn't been run yet
def evaluate(test_ds, predicted_ds):
    '''
    Function for evaluating MAE and RMSE between the predicted ratings and the test ratings.
    '''
    mask_test_ds = test_ds > 0
    MAE = np.sum(np.abs(test_ds[mask_test_ds] - predicted_ds[mask_test_ds])) / np.sum(mask_test_ds.astype(np.float32))
    RMSE = np.sqrt(np.sum(np.square(test_ds[mask_test_ds] - predicted_ds[mask_test_ds])) / np.sum(mask_test_ds.astype(np.float32)))
    return MAE, RMSE

# Step 1: Initialize the predicted rating matrix
predicted_ds = np.zeros((n_users, n_items))

# Step 2: Compute average rating for each item from the training dataset
item_avg_ratings = np.zeros(n_items)

for item_id in range(n_items):
    rated_by_users = train_ds[:, item_id] > 0
    if np.sum(rated_by_users) > 0:
        item_avg_ratings[item_id] = np.mean(train_ds[rated_by_users, item_id])
    else:
        item_avg_ratings[item_id] = 0  # If no ratings, set average to 0

# Step 3: Use item averages to predict test ratings
for user_id in range(n_users):
    for item_id in range(n_items):
        if test_ds[user_id, item_id] > 0:
            predicted_ds[user_id, item_id] = item_avg_ratings[item_id]

# Step 4: Evaluate prediction accuracy
MAE_solution2, RMSE_solution2 = evaluate(test_ds, predicted_ds)

# Print results (optional)
print("MAE (Method 2 - Item Average):", MAE_solution2)
print("RMSE (Method 2 - Item Average):", RMSE_solution2)


MAE (Method 2 - Item Average): 0.7961203950990416
RMSE (Method 2 - Item Average): 1.0013142101585724


In [9]:
# Please don't change this cell

print("===================== The MAE and RMSE of Your Implementation =====================")
print("MAE: {}, RMSE: {}" .format(MAE_solution2, RMSE_solution2))

===================== The MAE and RMSE of Your Implementation =====================
MAE: 0.7961203950990416, RMSE: 1.0013142101585724


# Your Solution for Method 3

In [8]:
# Write your code here for Method 3
# You are required to implement the required solution 1 here. 
# Then, evaluate your implementation by predicting the ratings in the test set (test_ds).
# Finally, save the corresponding MAE and RMSE of your implementation 
# into the following defined corresponding variable. 

MAE_solution3 = 0 # 0 is an intial value, you need to update this with the actual perofrmance of your implementation.
RMSE_solution3 = 0 # 0 is an intial value, you need to update this with the actual perofrmance of your implementation.
import numpy as np
from numpy.linalg import norm

# Use these if not already defined
n_users = train_ds.shape[0]
n_items = train_ds.shape[1]

# Evaluation function
def evaluate(test_ds, predicted_ds):
    mask_test_ds = test_ds > 0
    MAE = np.sum(np.abs(test_ds[mask_test_ds] - predicted_ds[mask_test_ds])) / np.sum(mask_test_ds.astype(np.float32))
    RMSE = np.sqrt(np.sum(np.square(test_ds[mask_test_ds] - predicted_ds[mask_test_ds])) / np.sum(mask_test_ds.astype(np.float32)))
    return MAE, RMSE

# Parameters
k = 5  # Number of neighbors

# Step 1: Precompute user average ratings
user_avg_ratings = np.zeros(n_users)
for user_id in range(n_users):
    rated_items = train_ds[user_id, :] > 0
    if np.sum(rated_items) > 0:
        user_avg_ratings[user_id] = np.mean(train_ds[user_id, rated_items])
    else:
        user_avg_ratings[user_id] = 0

# Step 2: Compute cosine similarity between users
user_similarity = np.zeros((n_users, n_users))
for u in range(n_users):
    for v in range(u + 1, n_users):  # only compute once for each pair
        u_ratings = train_ds[u, :]
        v_ratings = train_ds[v, :]
        mask = (u_ratings > 0) & (v_ratings > 0)
        if np.any(mask):
            u_vec = u_ratings[mask]
            v_vec = v_ratings[mask]
            sim = np.dot(u_vec, v_vec) / (norm(u_vec) * norm(v_vec) + 1e-9)
            user_similarity[u, v] = sim
            user_similarity[v, u] = sim  # symmetric

# Step 3: Predict ratings
predicted_ds = np.zeros((n_users, n_items))

for u in range(n_users):
    for i in range(n_items):
        if test_ds[u, i] > 0:
            # Find k similar users who rated item i
            neighbors = []
            for v in range(n_users):
                if u != v and train_ds[v, i] > 0:
                    neighbors.append((v, user_similarity[u, v]))

            neighbors.sort(key=lambda x: x[1], reverse=True)
            neighbors = neighbors[:k]

            if len(neighbors) > 0:
                num = 0
                denom = 0
                for v, sim in neighbors:
                    num += sim * train_ds[v, i]
                    denom += abs(sim)
                if denom > 0:
                    predicted_ds[u, i] = num / denom
                else:
                    predicted_ds[u, i] = user_avg_ratings[u]
            else:
                predicted_ds[u, i] = user_avg_ratings[u]

# Step 4: Evaluate
MAE_solution3, RMSE_solution3 = evaluate(test_ds, predicted_ds)

# Output the results
print("MAE (Method 3 - User-KNN):", MAE_solution3)
print("RMSE (Method 3 - User-KNN):", RMSE_solution3)


MAE (Method 3 - User-KNN): 0.8363910261857872
RMSE (Method 3 - User-KNN): 1.06608596971998


In [9]:
# Please don't change this cell

print("===================== The MAE and RMSE of Your Implementation =====================")
print("MAE: {}, RMSE: {}" .format(MAE_solution3, RMSE_solution3))

===================== The MAE and RMSE of Your Implementation =====================
MAE: 0.8363910261857872, RMSE: 1.06608596971998


# Your Solution for Method 4

In [10]:
# Write your code here for Method 4
# You are required to implement the required solution 1 here. 
# Then, evaluate your implementation by predicting the ratings in the test set (test_ds).
# Finally, save the corresponding MAE and RMSE of your implementation 
# into the following defined corresponding variable. 

MAE_solution4 = 0 # 0 is an intial value, you need to update this with the actual perofrmance of your implementation.
RMSE_solution4 = 0 # 0 is an intial value, you need to update this with the actual perofrmance of your implementation.
import numpy as np
from numpy.linalg import norm

# Ensure that train_ds, test_ds, n_users, and n_items are already defined from previous cells.

# Evaluation function (if not already defined)
def evaluate(test_ds, predicted_ds):
    mask_test_ds = test_ds > 0
    MAE = np.sum(np.abs(test_ds[mask_test_ds] - predicted_ds[mask_test_ds])) / np.sum(mask_test_ds.astype(np.float32))
    RMSE = np.sqrt(np.sum(np.square(test_ds[mask_test_ds] - predicted_ds[mask_test_ds])) / np.sum(mask_test_ds.astype(np.float32)))
    return MAE, RMSE

# Parameters
k = 5  # number of nearest neighbors

# Step 1: Precompute item average ratings
item_avg_ratings = np.zeros(n_items)
for item_id in range(n_items):
    rated_by_users = train_ds[:, item_id] > 0
    if np.sum(rated_by_users) > 0:
        item_avg_ratings[item_id] = np.mean(train_ds[rated_by_users, item_id])
    else:
        item_avg_ratings[item_id] = 0

# Step 2: Compute item-item cosine similarity matrix
item_similarity = np.zeros((n_items, n_items))
for i in range(n_items):
    for j in range(i + 1, n_items):
        i_ratings = train_ds[:, i]
        j_ratings = train_ds[:, j]
        mask = (i_ratings > 0) & (j_ratings > 0)
        if np.any(mask):
            i_vec = i_ratings[mask]
            j_vec = j_ratings[mask]
            sim = np.dot(i_vec, j_vec) / (norm(i_vec) * norm(j_vec) + 1e-9)
            item_similarity[i, j] = sim
            item_similarity[j, i] = sim  # symmetric

# Step 3: Predict ratings
predicted_ds = np.zeros((n_users, n_items))

for u in range(n_users):
    for i in range(n_items):
        if test_ds[u, i] > 0:
            # Find k similar items rated by user u
            neighbors = []
            for j in range(n_items):
                if i != j and train_ds[u, j] > 0:
                    neighbors.append((j, item_similarity[i, j]))

            neighbors.sort(key=lambda x: x[1], reverse=True)
            neighbors = neighbors[:k]

            if len(neighbors) > 0:
                num = 0
                denom = 0
                for j, sim in neighbors:
                    num += sim * train_ds[u, j]
                    denom += abs(sim)
                if denom > 0:
                    predicted_ds[u, i] = num / denom
                else:
                    predicted_ds[u, i] = item_avg_ratings[i]
            else:
                predicted_ds[u, i] = item_avg_ratings[i]

# Step 4: Evaluate
MAE_solution4, RMSE_solution4 = evaluate(test_ds, predicted_ds)

# Output the results
print("MAE (Method 4 - Item-KNN):", MAE_solution4)
print("RMSE (Method 4 - Item-KNN):", RMSE_solution4)


MAE (Method 4 - Item-KNN): 0.9205933038978861
RMSE (Method 4 - Item-KNN): 1.1678381520937895


In [11]:
# Please don't change this cell

print("===================== The MAE and RMSE of Your Implementation =====================")
print("MAE: {}, RMSE: {}" .format(MAE_solution4, RMSE_solution4))

===================== The MAE and RMSE of Your Implementation =====================
MAE: 0.9205933038978861, RMSE: 1.1678381520937895


# Your Solution for Method 5

In [12]:
# Write your code here for Method 5
# You are required to implement the required solution 1 here. 
# Then, evaluate your implementation by predicting the ratings in the test set (test_ds).
# Finally, save the corresponding MAE and RMSE of your implementation 
# into the following defined corresponding variable. 

MAE_solution5 = 0 # 0 is an intial value, you need to update this with the actual perofrmance of your implementation.
RMSE_solution5 = 0 # 0 is an intial value, you need to update this with the actual perofrmance of your implementation.
import numpy as np
from numpy.linalg import norm

# Assuming train_ds, test_ds, n_users, n_items, evaluate() function,
# user_similarity, item_similarity, user_avg_ratings, item_avg_ratings are defined.

# Number of neighbors
k = 5
# Weight parameter for combining predictions (between 0 and 1)
_lambda = 0.5

# --- Recompute user average ratings (if needed) ---
user_avg_ratings = np.zeros(n_users)
for user_id in range(n_users):
    rated_items = train_ds[user_id, :] > 0
    if np.sum(rated_items) > 0:
        user_avg_ratings[user_id] = np.mean(train_ds[user_id, rated_items])
    else:
        user_avg_ratings[user_id] = 0

# --- Recompute item average ratings (if needed) ---
item_avg_ratings = np.zeros(n_items)
for item_id in range(n_items):
    rated_by_users = train_ds[:, item_id] > 0
    if np.sum(rated_by_users) > 0:
        item_avg_ratings[item_id] = np.mean(train_ds[rated_by_users, item_id])
    else:
        item_avg_ratings[item_id] = 0

# --- Recompute user similarity matrix (cosine similarity) ---
user_similarity = np.zeros((n_users, n_users))
for u in range(n_users):
    for v in range(u + 1, n_users):
        u_ratings = train_ds[u, :]
        v_ratings = train_ds[v, :]
        mask = (u_ratings > 0) & (v_ratings > 0)
        if np.any(mask):
            u_vec = u_ratings[mask]
            v_vec = v_ratings[mask]
            sim = np.dot(u_vec, v_vec) / (norm(u_vec) * norm(v_vec) + 1e-9)
            user_similarity[u, v] = sim
            user_similarity[v, u] = sim

# --- Recompute item similarity matrix (cosine similarity) ---
item_similarity = np.zeros((n_items, n_items))
for i in range(n_items):
    for j in range(i + 1, n_items):
        i_ratings = train_ds[:, i]
        j_ratings = train_ds[:, j]
        mask = (i_ratings > 0) & (j_ratings > 0)
        if np.any(mask):
            i_vec = i_ratings[mask]
            j_vec = j_ratings[mask]
            sim = np.dot(i_vec, j_vec) / (norm(i_vec) * norm(j_vec) + 1e-9)
            item_similarity[i, j] = sim
            item_similarity[j, i] = sim

# --- Prediction ---
predicted_ds = np.zeros((n_users, n_items))

for u in range(n_users):
    for i in range(n_items):
        if test_ds[u, i] > 0:
            # --- User-KNN prediction ---
            neighbors_u = []
            for v in range(n_users):
                if u != v and train_ds[v, i] > 0:
                    neighbors_u.append((v, user_similarity[u, v]))
            neighbors_u.sort(key=lambda x: x[1], reverse=True)
            neighbors_u = neighbors_u[:k]

            if len(neighbors_u) > 0:
                num_u = 0
                denom_u = 0
                for v, sim in neighbors_u:
                    num_u += sim * train_ds[v, i]
                    denom_u += abs(sim)
                pred_user = num_u / denom_u if denom_u > 0 else user_avg_ratings[u]
            else:
                pred_user = user_avg_ratings[u]

            # --- Item-KNN prediction ---
            neighbors_i = []
            for j in range(n_items):
                if i != j and train_ds[u, j] > 0:
                    neighbors_i.append((j, item_similarity[i, j]))
            neighbors_i.sort(key=lambda x: x[1], reverse=True)
            neighbors_i = neighbors_i[:k]

            if len(neighbors_i) > 0:
                num_i = 0
                denom_i = 0
                for j, sim in neighbors_i:
                    num_i += sim * train_ds[u, j]
                    denom_i += abs(sim)
                pred_item = num_i / denom_i if denom_i > 0 else item_avg_ratings[i]
            else:
                pred_item = item_avg_ratings[i]

            # --- Combine predictions ---
            predicted_ds[u, i] = _lambda * pred_user + (1 - _lambda) * pred_item

# --- Evaluate ---
MAE_solution5, RMSE_solution5 = evaluate(test_ds, predicted_ds)

# Output results
print("MAE (Method 5 - Hybrid User+Item KNN):", MAE_solution5)
print("RMSE (Method 5 - Hybrid User+Item KNN):", RMSE_solution5)


MAE (Method 5 - Hybrid User+Item KNN): 0.8073362177721886
RMSE (Method 5 - Hybrid User+Item KNN): 1.0146392974108553


In [13]:
# Please don't change this cell

print("===================== The MAE and RMSE of Your Implementation =====================")
print("MAE: {}, RMSE: {}" .format(MAE_solution5, RMSE_solution5))

===================== The MAE and RMSE of Your Implementation =====================
MAE: 0.8073362177721886, RMSE: 1.0146392974108553
